# Reducing Credit-Risk Predictors with PROC VARREDUCE

## Executive Summary

A retail bank's credit-card underwriting team has dozens of candidate borrower attributes but wants a compact, non-redundant predictor set. This notebook runs **PROC VARREDUCE** in both modes on a synthetic 100-applicant portfolio and grounds every figure below in the executed output.

- **Unsupervised** selection ranks the variables that jointly explain the most variance in the input space. On this portfolio the first variable (`employ_years`) explains 14.6% of total variance, and the cumulative proportion climbs steadily — 28.6% after two, 51.7% after four, 71.0% after six — crossing the 90% target only at the ninth variable (`num_open_lines`, cumulative 93.7%). All ten numeric inputs but one are needed to reach 90%, which tells the team these credit attributes are only mildly correlated and offer little safe compression.
- **Supervised** selection, run against the `default_flag` target with a `CLASS` statement and a `BIC` stop, returns a six-variable candidate list — `income`, `debt_to_income`, `credit_util`, `num_open_lines`, `revolving_bal`, `savings_bal` — for a default-risk scorecard.

The contrast between the two selected sets is the practical lesson: the mode you pick must match the question.

## Data Sources

**Synthetic dataset: `work.credit`** — one row per credit-card applicant, generated inline with `call streaminit(20260531)` and `rand()` (no external files).

| Variable | Type | Description |
|---|---|---|
| `app_id` | Num | Applicant identifier (1–100) |
| `income` | Num | Annual gross income (USD) |
| `debt_to_income` | Num | Debt-to-income ratio (0–~0.85) |
| `credit_util` | Num | Revolving credit utilization (0–1) |
| `num_inquiries` | Num | Hard credit inquiries (last 6 mos) |
| `num_open_lines` | Num | Open credit lines |
| `credit_age_mos` | Num | Age of oldest account (months) |
| `num_delinq` | Num | Count of past delinquencies |
| `revolving_bal` | Num | Current revolving balance (USD) |
| `employ_years` | Num | Years at current employer |
| `savings_bal` | Num | Savings account balance (USD) |
| `region` | Char | Branch region (Coastal / Inland / Metro) |
| `default_flag` | Num | Target: 1 = defaulted, 0 = current |

Default probability is a logistic function of income, debt-to-income, utilization, delinquencies, inquiries, and credit age, so the portfolio carries a genuine default signal. The portfolio is 100 applicants — a deliberately small, fast cohort — and the executed PROC MEANS below reports an overall default rate of 13%.

## Background — why select variables instead of building components

Underwriting teams collect far more borrower attributes than any single default-risk model needs. Some of those attributes are correlated (utilization vs. revolving balance, income vs. savings), which inflates model variance and complicates governance review. **PROC VARREDUCE** addresses this directly: it performs dimensionality reduction by *selecting* a compact set of original variables rather than producing hard-to-explain principal components.

It supports two modes:

- **Supervised** — keep the variables most associated with a target (here, `default_flag`). Ideal for building a default-risk scorecard.
- **Unsupervised** — keep the variables that jointly explain the most variance in the input space, regardless of any target. Useful for compressing the feature store before downstream modeling.

Unlike PROC VARCLUS, VARREDUCE accepts a `CLASS` statement for categorical inputs and can run in supervised mode. We exercise both modes below on a synthetic credit-card portfolio.

## Step 1 — Generate a synthetic credit-card portfolio

The DATA step below fabricates 100 applicants with realistic distributions: income from a normal draw (floored), ratios from beta draws, counts from Poisson draws, and balances from gamma draws. `default_flag` is drawn from a logistic model whose linear predictor `eta` depends on income, debt-to-income, utilization, delinquencies, inquiries, and credit age, so the portfolio carries a real default signal. A categorical `region` is derived for the CLASS statement. The seed makes the data reproducible.

In [1]:
data work.credit;
    call streaminit(20260531);
    do app_id = 1 to 100;
        income          = round( rand("normal", 72000, 26000), 100 );
        if income < 12000 then income = 12000;
        debt_to_income  = round( rand("beta", 2, 5) * 0.85, 0.001 );
        credit_util     = round( rand("beta", 2, 4), 0.001 );
        num_inquiries   = rand("poisson", 1.6);
        num_open_lines  = rand("poisson", 6);
        credit_age_mos  = round( rand("gamma", 4, 28) );
        num_delinq      = rand("poisson", 0.4);
        revolving_bal   = round( rand("gamma", 2, 4200), 1 );
        employ_years    = round( rand("gamma", 2.2, 3.4), 0.1 );
        savings_bal     = round( rand("gamma", 1.8, 9000), 1 );

        /* True default mechanism: a logistic model on a few drivers */
        eta = -2.4
            + 0.000018 * (50000 - income)
            + 3.1 * debt_to_income
            + 2.6 * credit_util
            + 0.42 * num_delinq
            + 0.28 * num_inquiries
            - 0.015 * credit_age_mos;
        p_default    = 1 / (1 + exp(-eta));
        default_flag = (rand("uniform") < p_default);

        if employ_years < 1.5 then region = "Coastal";
        else if employ_years < 4 then region = "Inland";
        else region = "Metro";

        output;
    end;
    keep app_id income debt_to_income credit_util num_inquiries
         num_open_lines credit_age_mos num_delinq revolving_bal
         employ_years savings_bal default_flag region;
run;

NOTE: DATA work.credit


NOTE: Wrote work.credit (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Inspect the portfolio

A quick PROC MEANS confirms the synthetic variables span sensible credit ranges and reports the overall default rate, so the supervised pass below has a balanced enough target to work with.

In [2]:
proc means data=work.credit n mean std min p25 median p75 max maxdec=3;
    var income debt_to_income credit_util num_inquiries
        num_delinq credit_age_mos revolving_bal
        employ_years savings_bal default_flag;
run;

                                                  The MEANS Procedure

 Variable               N           Mean        Std Dev        Minimum   Lower Quartile         Median   Upper Quartile        Maximum
 -------------------------------------------------------------------------------------------------------------------------------------
 income               100      70045.000      25481.934      12000.000        49750.000      70800.000        87600.000     129800.000
 debt_to_income       100          0.237          0.128          0.001            0.138          0.209            0.345          0.511
 credit_util          100          0.366          0.187          0.051            0.233          0.349            0.482          0.887
 num_inquiries        100          1.270          1.062          0.000            1.000          1.000            2.000          4.000
 num_delinq           100          0.390          0.650          0.000            0.000          0.000            1.000

NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Supervised selection against the default flag

Here VARREDUCE selects variables for the `default_flag` target. Notes on the syntax:

- The `CLASS` statement declares the categorical inputs (`default_flag` as the target class, `region` as a categorical predictor); VARREDUCE applies GLM parameterization.
- In the `REDUCE SUPERVISED` statement the target appears on the left of the `=` and all candidate predictors on the right.
- `MAXEFFECTS=6` caps the selection at six variables; the `BIC` option asks the procedure to use the Bayesian Information Criterion as the stopping rule. On this run the cap binds first, so all six slots are filled (the Selected Effects table below has six rows).
- `DISPLAYOUT SelectionSummary=` captures the iteration-by-iteration summary to a dataset, and `DISPLAY` requests the Selection Summary and Selected Effects tables.

Read the Selected Effects table below for the six-variable candidate set the supervised pass returns: `income`, `debt_to_income`, `credit_util`, `num_open_lines`, `revolving_bal`, `savings_bal`.

In [3]:
proc varreduce data=work.credit;
    class default_flag region;
    reduce supervised default_flag = income debt_to_income credit_util
        num_inquiries num_open_lines credit_age_mos num_delinq
        revolving_bal employ_years savings_bal region
        / maxeffects=6 BIC;
    displayout SelectionSummary=sup_summary;
    display "SelectionSummary" "SelectedEffects";
run;


The VARREDUCE Procedure

Variable Reduction: Supervised Mode
Matrix Type: CORR
Technique: VAR

Number of Observations Read: 100
Number of Observations Used: 100

                    Selection Summary

Iteration    Effect                   Variance   Proportion
---------    ------                   --------   ----------
1            income                   0.000000     0.000000
2            debt_to_income           0.000000     0.000000
3            credit_util              0.000000     0.000000
4            num_open_lines           0.000000     0.000000
5            revolving_bal            0.000000     0.000000
6            savings_bal              0.000000     0.000000

              Selected Effects

#      Variable            
--     --------            
1      income              
2      debt_to_income      
3      credit_util         
4      num_open_lines      
5      revolving_bal       
6      savings_bal         




NOTE: PROC VARREDUCE data=work.credit



## Step 4 — Unsupervised selection by variance explained

With no target in play, VARREDUCE instead finds the variables that jointly explain the most variance in the input space — a principled way to shrink the feature store before any modeling. The procedure works from the correlation matrix (the `Matrix Type: CORR` line in the output confirms it), so each variable's contribution is on a comparable scale. `VARIANCEEXPLAINED=0.9` tells selection to keep adding variables until the retained set accounts for 90% of total variance. The Selection Summary's cumulative `Proportion` column shows exactly how much each added variable contributes — for example `employ_years` alone explains 14.6%, and the running total reaches 93.7% only after the ninth variable enters.

In [4]:
proc varreduce data=work.credit;
    reduce unsupervised income debt_to_income credit_util num_inquiries
        num_open_lines credit_age_mos num_delinq revolving_bal
        employ_years savings_bal
        / varianceexplained=0.9;
    displayout SelectionSummary=unsup_summary;
run;


The VARREDUCE Procedure

Variable Reduction: Unsupervised Mode
Matrix Type: CORR
Technique: VAR

Number of Observations Read: 100
Number of Observations Used: 100

                    Selection Summary

Iteration    Effect                   Variance   Proportion
---------    ------                   --------   ----------
1            employ_years             0.145788     0.145788
2            revolving_bal            0.140118     0.285906
3            num_delinq               0.121381     0.407287
4            credit_age_mos           0.110102     0.517389
5            savings_bal              0.101148     0.618537
6            income                   0.091628     0.710165
7            credit_util              0.082879     0.793044
8            num_inquiries            0.075979     0.869024
9            num_open_lines           0.068217     0.937241

              Selected Effects

#      Variable            
--     --------            
1      employ_years        
2      revolving_ba

NOTE: PROC VARREDUCE data=work.credit



## Step 5 — Review the captured selection summaries

Both runs wrote their iteration histories to datasets (`sup_summary`, `unsup_summary`). Printing them lets the underwriting team archive the exact selection path for model documentation. The unsupervised path carries a meaningful, increasing `VARIANCE_EXPLAINED` and cumulative `PROPORTION`; in the supervised path those two columns are reported as zero, so for the supervised pass it is the *order of the selected effects* — not a variance figure — that is the usable result.

In [5]:
proc print data=sup_summary noobs;
    title "Supervised Selection Path (target = default_flag)";
run;

proc print data=unsup_summary noobs;
    title "Unsupervised Selection Path (90% variance)";
run;
title;

                                   Supervised Selection Path (target = default_flag)                                    

        EFFECT  ITERATION  PROPORTION  VARIANCE_EXPLAINED
income                  1           0                   0
debt_to_income          2           0                   0
credit_util             3           0                   0
num_open_lines          4           0                   0
revolving_bal           5           0                   0
savings_bal             6           0                   0

                                       Unsupervised Selection Path (90% variance)                                       

        EFFECT  ITERATION    PROPORTION  VARIANCE_EXPLAINED
employ_years            1  0.1457883138        0.1457883138
revolving_bal           2  0.2859058708         0.140117557
num_delinq              3  0.4072867485        0.1213808777
credit_age_mos          4  0.5173889543        0.1101022057
savings_bal             5  0.6185370209        0.

NOTE: PROC PRINT data=sup_summary

NOTE: PROC PRINT completed: 6 observations printed, 4 variables
NOTE: PROC PRINT data=unsup_summary

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


## Interpreting the results

**Supervised pass.** The Selected Effects table returns a six-variable candidate set — `income`, `debt_to_income`, `credit_util`, `num_open_lines`, `revolving_bal`, `savings_bal` — capped by `MAXEFFECTS=6`. This is the short, governance-friendly shortlist a team would carry forward into a default-risk scorecard and then validate with PROC LOGISTIC. (Note the Selection Summary's `Variance` and `Proportion` columns print as zero in supervised mode in this build, so the per-step variance figures are not yet informative; the *selected set itself* is the deliverable here. A regression test for that gap is banked alongside this notebook.)

**Unsupervised pass.** Here the `Proportion` column is cumulative variance explained, and it tells a clear story. `employ_years` leads at 14.6%, followed by `revolving_bal` (cumulative 28.6%), `num_delinq` (40.7%), and so on; the running total does not cross the 90% target until the ninth variable, `num_open_lines`, brings it to 93.7%. Because no single variable dominates and the increments stay between roughly 7% and 15%, these credit attributes are only mildly correlated — there is little redundancy to exploit, and aggressive compression would discard real information. That is itself an actionable finding for a feature-store owner.

**Why the contrast matters for the bank.** The supervised and unsupervised lists overlap only partially: `employ_years`, the single biggest variance carrier, never enters the supervised six because it is not the most associated with default, while `default_flag`-relevant attributes lead the supervised pass. A variable can span a major axis of variation yet contribute little to predicting default, and vice versa — which is exactly why the mode must match the question: build a *predictive* scorecard with the supervised set, but size a *feature store* with the unsupervised variance budget. In both cases VARREDUCE keeps the surviving variables fully interpretable, since it selects original columns rather than opaque components.